## combination of my scraper and romy's
This tests the first 50 people from the json file created by scraper 1. 


In [1]:
!pip install requests beautifulsoup4 tqdm

In [3]:
import json
import time
import random
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime
import os

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE = "../data/raw/hyrox_results_2026-11-24_10-20.json"

# ── Adjust this per computer ─────────────────
START = 0        
# ← start index
END   = 50   
# ← end index
# ─────────────────────────────────────────────

OUTPUT_FILE     = f"../data/raw/hyrox_detailed_{START}_{END}.json"
CHECKPOINT_FILE = f"../data/raw/hyrox_checkpoint_{START}_{END}.json"
ERROR_FILE      = f"../data/raw/hyrox_errors_{START}_{END}.json"

# CSS classes for workout stations only (from teammate's code)
WANTED_CLASSES = (
    [f"f-time_{i}" for i in range(11, 19)]
    + ["f-time_49", "f-time_50", "f-time_60"]
)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
]

def get_headers():
    return {"User-Agent": random.choice(USER_AGENTS)}

# ─────────────────────────────────────────────
# SCRAPE FUNCTION
# ─────────────────────────────────────────────
def scrape_personal_page(url: str) -> dict:
    result = {
        "judging": {},
        "workout_summary": {},
        "scrape_error": None
    }

    
# ── Request with retry ───────────────────
    for attempt in range(3):
        try:
            response = requests.get(url, headers=get_headers(), timeout=10)
            if response.status_code == 403:
                time.sleep(random.uniform(10, 20))
                continue
            response.raise_for_status()
            break
        except requests.RequestException as e:
            if attempt == 2:
                result["scrape_error"] = str(e)
                return result
            time.sleep(random.uniform(5, 10))

    soup = BeautifulSoup(response.text, "html.parser")

    
# ── 1. Judging Decision ──────────────────
    judges_div = soup.find("div", id="detail-box-judges")
    if judges_div:
        for row in judges_div.find_all("tr"):
            label    = row.find("th", class_="desc")
            value_td = row.find("td")
            if label and value_td:
                key   = label.get_text(strip=True).replace("*", "").lower().replace(" ", "_")
                muted = value_td.find("span", class_="text-muted")
                value = None if muted else value_td.get_text(strip=True)
                result["judging"][key] = value
    else:
        result["scrape_error"] = "Judging section not found"

    
# ── 2. Workout Summary ───────────────────
    
# Using teammate's cleaner selector: #detail-box-other table
    
# Combined with our WANTED_CLASSES filter to get stations only
    table = soup.select_one("#detail-box-other table.table")
    if table:
        for row in table.select("tbody tr"):
            row_classes = row.get("class", [])

            
# Filter on wanted classes (teammate's approach)
            if any(cls in row_classes for cls in WANTED_CLASSES):
                cells = row.find_all(["th", "td"], recursive=False)
                if len(cells) == 3:
                    split_name = cells[0].get_text(strip=True)
                    time_val   = cells[1].get_text(strip=True)
                    place_raw  = cells[2].get_text(strip=True)

                    
# Convert "–" to None (our approach)
                    muted    = cells[2].find("span", class_="text-muted")
                    place_val = None if muted else place_raw

                    result["workout_summary"][split_name] = {
                        "time": time_val,
                        "place": int(place_val) if place_val and place_val.isdigit() else None
                    }
    else:
        result["scrape_error"] = "Workout table not found"

    return result

# ─────────────────────────────────────────────
# CHECKPOINT HELPERS
# ─────────────────────────────────────────────
def load_checkpoint() -> set:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return set(json.load(f))
    return set()

def save_checkpoint(done_ids: set):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(list(done_ids), f)

def load_existing_results() -> list:
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

# ─────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────
def main():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        all_athletes = json.load(f)

    athletes = all_athletes[START:END]

    print(f"Total loaded    : {len(all_athletes)} athletes")
    print(f"Your batch      : [{START} → {END}] ({len(athletes)} athletes)")

    done_ids         = load_checkpoint()
    detailed_results = load_existing_results()
    errors           = []

    remaining = [a for a in athletes if a.get("record_id") not in done_ids]
    print(f"Already done    : {len(done_ids)}")
    print(f"Remaining       : {len(remaining)}\n")

    for athlete in tqdm(remaining, desc=f"Batch {START}-{END}", unit="athlete"):
        url       = athlete.get("personal_link")
        record_id = athlete.get("record_id")

        if not url:
            athlete["detail_data"] = None
            detailed_results.append(athlete)
            done_ids.add(record_id)
            continue

        detail = scrape_personal_page(url)

        if detail["scrape_error"]:
            errors.append({
                "record_id": record_id,
                "url"      : url,
                "error"    : detail["scrape_error"],
                "timestamp": datetime.now().isoformat()
            })

        athlete["detail_data"] = {
            "judging"        : detail["judging"],
            "workout_summary": detail["workout_summary"]
        }
        detailed_results.append(athlete)
        done_ids.add(record_id)

        
# Save checkpoint every 50 athletes
        if len(done_ids) % 50 == 0:
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(detailed_results, f, ensure_ascii=False, indent=2)
            save_checkpoint(done_ids)

        time.sleep(random.uniform(2.0, 5.0))

    
# Final save
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(detailed_results, f, ensure_ascii=False, indent=2)
    save_checkpoint(done_ids)

    print(f"\nDone! Saved to '{OUTPUT_FILE}'")

    if errors:
        with open(ERROR_FILE, "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)
        print(f" {len(errors)} errors saved to '{ERROR_FILE}'")
    else:
        print("No errors!")

if __name__ == "__main__":
    main()

main()


Total loaded    : 20070 athletes
Your batch      : [0 → 50] (50 athletes)
Already done    : 50
Remaining       : 0



Batch 0-50: 0athlete [00:00, ?athlete/s]



Done! Saved to '../data/raw/hyrox_detailed_0_50.json'
No errors!
Total loaded    : 20070 athletes
Your batch      : [0 → 50] (50 athletes)
Already done    : 50
Remaining       : 0



Batch 0-50: 0athlete [00:00, ?athlete/s]


Done! Saved to '../data/raw/hyrox_detailed_0_50.json'
No errors!
